# Notebook 5: Error Handling and Debugging

CUDA errors are silent by default — your program keeps running with garbage results. In this notebook, you will learn:

- CUDA error checking with `cudaGetLastError()` and return values
- Writing a reusable error-checking macro
- Common CUDA errors and how to diagnose them
- Debugging strategies

In [ ]:
%load_ext nvcc4jupyter

## 5.1 The Problem: Silent Failures

CUDA API calls return error codes, but kernel launches don't. If you don't check, errors go unnoticed.

In [ ]:
%%cuda
#include <cstdio>

__global__ void bad_kernel(int* data) {
    // Deliberately access invalid memory
    data[threadIdx.x] = threadIdx.x;  // data is NULL!
}

int main() {
    // Launch with NULL pointer — this will fail
    bad_kernel<<<1, 32>>>(nullptr);

    // Without error checking, we have no idea it failed!
    printf("Program finished 'successfully'\n");

    // But if we check...
    cudaError_t err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        printf("CUDA error: %s\n", cudaGetErrorString(err));
    }

    return 0;
}

## 5.2 Two Kinds of Errors

### Synchronous errors (API calls)
Functions like `cudaMalloc`, `cudaMemcpy` return `cudaError_t` directly:
```cpp
cudaError_t err = cudaMalloc(&ptr, bytes);
if (err != cudaSuccess) {
    printf("Error: %s\n", cudaGetErrorString(err));
}
```

### Asynchronous errors (kernel launches)
Kernels launch asynchronously — errors surface later. Check with:
```cpp
kernel<<<grid, block>>>(args);
cudaError_t err = cudaGetLastError();        // Check launch config errors
cudaError_t err2 = cudaDeviceSynchronize();  // Check execution errors
```

## 5.3 The CUDA_CHECK Macro

The standard practice is to define a macro that checks every CUDA call. This is the single most important debugging tool.

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>

// The essential error-checking macro
#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d — %s\n",                     \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

// Kernel launch checker
#define CUDA_CHECK_KERNEL()                                                    \
    do {                                                                       \
        CUDA_CHECK(cudaGetLastError());                                        \
        CUDA_CHECK(cudaDeviceSynchronize());                                   \
    } while (0)

__global__ void vector_add(const float* a, const float* b, float* c, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) c[idx] = a[idx] + b[idx];
}

int main() {
    const int N = 1024;
    const size_t bytes = N * sizeof(float);

    float *d_a, *d_b, *d_c;

    // Every CUDA call is wrapped with CUDA_CHECK
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));

    // Initialize with zeros for demo
    CUDA_CHECK(cudaMemset(d_a, 0, bytes));
    CUDA_CHECK(cudaMemset(d_b, 0, bytes));

    // Launch and check
    vector_add<<<(N + 255) / 256, 256>>>(d_a, d_b, d_c, N);
    CUDA_CHECK_KERNEL();

    printf("All CUDA operations completed successfully!\n");

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return 0;
}

### Why this matters

Without `CUDA_CHECK`, a failed `cudaMalloc` silently gives you a NULL pointer, and the kernel writes to NULL, and `cudaMemcpy` copies garbage, and you spend hours debugging the wrong thing.

**Rule: Wrap every CUDA API call with `CUDA_CHECK` in all your code.**

## 5.4 Common CUDA Errors

| Error | Typical Cause |
|-------|---------------|
| `cudaErrorMemoryAllocation` | Not enough GPU memory. Check with `nvidia-smi`. |
| `cudaErrorInvalidConfiguration` | Bad launch parameters (e.g., >1024 threads/block). |
| `cudaErrorInvalidValue` | Bad argument (NULL pointer, negative size). |
| `cudaErrorIllegalAddress` | Kernel accessed invalid memory (out of bounds). |
| `cudaErrorLaunchTimeout` | Kernel took too long (display timeout, ~2-5 seconds). |
| `cudaErrorNoDevice` | No CUDA-capable GPU found. |

Let's trigger some of these intentionally:

In [ ]:
%%cuda
#include <cstdio>

#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            printf("CUDA error at line %d: %s\n",                              \
                    __LINE__, cudaGetErrorString(err));                        \
        }                                                                      \
    } while (0)

__global__ void dummy() {}

int main() {
    // Error 1: Allocate too much memory
    float* ptr;
    printf("--- Allocating 999 TB ---\n");
    CUDA_CHECK(cudaMalloc(&ptr, (size_t)999 * 1024 * 1024 * 1024 * 1024));

    // Error 2: Too many threads per block
    printf("\n--- Launching 2048 threads per block ---\n");
    dummy<<<1, 2048>>>();
    CUDA_CHECK(cudaGetLastError());

    // Error 3: Zero blocks (no work)
    printf("\n--- Launching 0 blocks ---\n");
    dummy<<<0, 256>>>();
    CUDA_CHECK(cudaGetLastError());

    // Reset error state
    cudaGetLastError();

    printf("\nDone — errors were caught and reported!\n");
    return 0;
}

## 5.5 Sticky vs Non-Sticky Errors

CUDA has two error classes:

- **Non-sticky errors** (e.g., `cudaErrorInvalidConfiguration`): Cleared after you read them with `cudaGetLastError()`.
- **Sticky errors** (e.g., `cudaErrorIllegalAddress`): Corrupt the CUDA context. The only recovery is destroying the context (restarting the program).

This is why it's critical to catch errors early — a sticky error makes all subsequent CUDA calls fail.

## 5.6 Debugging Strategy: Serial Execution

The grid-stride loop pattern lets you debug by launching just one thread:

```cpp
// Normal launch: parallel
kernel<<<(N+255)/256, 256>>>(data, N);

// Debug launch: serial (1 thread processes everything)
kernel<<<1, 1>>>(data, N);
```

With 1 thread, you can add `printf` statements and get sequential output.

In [ ]:
%%cuda
#include <cstdio>

__global__ void debug_kernel(float* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;

    for (int i = idx; i < N; i += stride) {
        // Debug print (only practical with very few elements or 1 thread)
        printf("  data[%d]: %.2f -> %.2f\n", i, data[i], data[i] * 2.0f);
        data[i] *= 2.0f;
    }
}

int main() {
    const int N = 8;
    float* data;
    cudaMallocManaged(&data, N * sizeof(float));
    for (int i = 0; i < N; i++) data[i] = (float)(i + 1);

    printf("Debug mode (1 thread, serial execution):\n");
    debug_kernel<<<1, 1>>>(data, N);
    cudaDeviceSynchronize();

    printf("\nResult: ");
    for (int i = 0; i < N; i++) printf("%.0f ", data[i]);
    printf("\n");

    cudaFree(data);
    return 0;
}

## 5.7 `cuda-memcheck` and `compute-sanitizer`

NVIDIA provides tools similar to Valgrind for CUDA:

```bash
# Modern (CUDA 11.6+)
compute-sanitizer --tool memcheck ./my_program

# Legacy
cuda-memcheck ./my_program
```

These detect:
- Out-of-bounds memory accesses
- Misaligned accesses
- Race conditions (`--tool racecheck`)
- Uninitialized memory reads (`--tool initcheck`)

**Warning:** Running under memcheck makes your program 10-100x slower. Use it for debugging, not benchmarking.

## 5.8 Defensive Programming Template

Here's a complete template with proper error handling that you should use as a starting point for all CUDA programs:

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cmath>

// ============================================================
// Error handling macros — include these in every CUDA program
// ============================================================
#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

#define CUDA_CHECK_KERNEL()                                                    \
    do {                                                                       \
        CUDA_CHECK(cudaGetLastError());                                        \
        CUDA_CHECK(cudaDeviceSynchronize());                                   \
    } while (0)

// ============================================================
// Your kernel
// ============================================================
__global__ void my_kernel(float* out, const float* in, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;
    for (int i = idx; i < N; i += stride) {
        out[i] = sqrtf(in[i]);
    }
}

int main() {
    const int N = 1 << 20;
    const size_t bytes = N * sizeof(float);

    // Allocate
    float* h_in = (float*)malloc(bytes);
    float* h_out = (float*)malloc(bytes);
    float *d_in, *d_out;
    CUDA_CHECK(cudaMalloc(&d_in, bytes));
    CUDA_CHECK(cudaMalloc(&d_out, bytes));

    // Initialize
    for (int i = 0; i < N; i++) h_in[i] = (float)(i + 1);

    // Transfer
    CUDA_CHECK(cudaMemcpy(d_in, h_in, bytes, cudaMemcpyHostToDevice));

    // Launch
    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;
    my_kernel<<<grid_size, block_size>>>(d_out, d_in, N);
    CUDA_CHECK_KERNEL();

    // Copy back
    CUDA_CHECK(cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost));

    // Verify
    int errors = 0;
    for (int i = 0; i < N; i++) {
        float expected = sqrtf((float)(i + 1));
        if (fabsf(h_out[i] - expected) > 1e-5) errors++;
    }
    printf("Verification: %s (%d errors out of %d)\n",
           errors == 0 ? "PASSED" : "FAILED", errors, N);

    // Cleanup
    CUDA_CHECK(cudaFree(d_in));
    CUDA_CHECK(cudaFree(d_out));
    free(h_in);
    free(h_out);
    return 0;
}

## 5.9 Key Takeaways

1. **CUDA errors are silent** unless you check for them
2. **`CUDA_CHECK` macro** — use it on every CUDA API call, no exceptions
3. **`CUDA_CHECK_KERNEL()`** after every kernel launch checks both launch and execution errors
4. **Sticky errors** corrupt the CUDA context — catch errors early
5. **Debug with `<<<1, 1>>>`** to get serial execution with printf
6. **`compute-sanitizer`** finds memory bugs like Valgrind does for CPU code
7. **Always verify correctness** against a CPU reference implementation

## Exercises

**Exercise 5.1:** Take the vector addition code from Notebook 4 and add full error checking with `CUDA_CHECK` on every CUDA call.

**Exercise 5.2:** Intentionally introduce a bug (e.g., allocating too little memory, wrong grid size, accessing out of bounds) and verify that `CUDA_CHECK` catches it.

**Exercise 5.3:** Write a kernel with a subtle bug: forget the bounds check. Run it with a data size that's not a multiple of the block size. Does it crash? Does `compute-sanitizer` catch it?

In [ ]:
%%cuda
// Exercise 5.2 — Trigger an error and catch it
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

__global__ void kernel(float* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    // TODO: Intentionally forget bounds check — what happens?
    data[idx] = 42.0f;
}

int main() {
    // TODO: Allocate less memory than needed and see what error we get
    return 0;
}

---
**Next:** [Notebook 6 — 2D Grids and Matrix Operations](06_2D_Grids_and_Matrix_Ops.ipynb) — Working with matrices and 2D indexing.